In [ ]:
# hide
# no-output
from IPython.utils.capture import capture_output
with capture_output():
    %pip install -q plotly anywidget

import asyncio
import os
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import Audio
import pyquist as pq
import icm_plotly
from icm_plotly import RED, BLUE, GOLD, IRON, TEAL, STEEL

Pick a transfer function and drag the drive $g$. Top left is the transfer
function $f(x)$, which amplifies its input by $g$ before shaping it, against
the dashed identity $f(x) = x$. Top right is two cycles of a full-scale
sinusoid $x[n]$ (dashed blue) and the waveshaped output $f(x[n])$. Bottom is
the output's spectrum: every bar past the first is a harmonic that the input
never had. All three functions are odd, $f(-x) = -f(x)$, so only odd
harmonics appear. The audio card plays the waveshaped sinusoid or guitar.

In [ ]:
# hide
# autorun
SHAPE0, DRIVE0, SOURCE0 = "Soft clipping", 3.0, "Sinusoid (220 Hz)"

GREEN = "#2CA02C"                   # the chapter's soft-clipping green
guitar = pq.Audio.from_file("../assets/guitar-acoustic.wav").as_mono()
SR = guitar.sample_rate
G = np.asarray(guitar.samples).reshape(-1)[:5 * SR]
SOURCES = {
    "Sinusoid (220 Hz)": np.sin(2 * np.pi * 220 * np.arange(2 * SR) / SR),
    "Acoustic guitar": G / np.abs(G).max(),
}
# each transfer function f(x), with its drive g, as the chapter writes it,
# in the chapter's colors (hard clipping red, soft clipping green)
SHAPES = {
    "Hard clipping": (lambda x, g: np.clip(g * x, -1.0, 1.0),
                      "min(1, max(&minus;1, <i>gx</i>))", RED),
    "Soft clipping": (lambda x, g: np.tanh(g * x), "tanh(<i>gx</i>)", GREEN),
    "Wavefolder": (lambda x, g: np.sin(g * x), "sin(<i>gx</i>)", GOLD),
}
XT = np.linspace(-1.0, 1.0, 401)                 # transfer-function axis
N_P = 4096                                       # one period, for the spectrum
SIN = np.sin(2 * np.pi * np.arange(N_P) / N_P)
PH = np.linspace(0.0, 2.0, 601)                  # two periods, for the plot
SIN_P = np.sin(2 * np.pi * PH)
K = np.arange(1, 26)                             # harmonics shown


def shape(name, g, x, SHAPES=SHAPES):
    return SHAPES[name][0](x, g)


def harmonics(y, N_P=N_P, K=K):
    """Amplitude of each harmonic of one period of y, in dB (1 = 0 dB)."""
    mag = np.abs(np.fft.rfft(y[:N_P])) * 2 / N_P
    return 20 * np.log10(np.maximum(mag[K], 1e-5))


def stems(n, v, base=-80.0):
    xs = np.repeat(np.asarray(n, float), 3)
    ys = np.column_stack([np.full(len(v), base), v, np.full(len(v), np.nan)]).ravel()
    xs[2::3] = np.nan
    return xs, ys


def figure():
    y = shape(SHAPE0, DRIVE0, SIN)
    yp = shape(SHAPE0, DRIVE0, SIN_P)
    color = SHAPES[SHAPE0][2]
    fig = make_subplots(rows=2, cols=2, specs=[[{}, {}], [{"colspan": 2}, None]],
                        vertical_spacing=0.2, horizontal_spacing=0.12)
    fig.add_scatter(x=XT, y=XT, mode="lines",
                    line=dict(color=IRON, width=1, dash="dash"), row=1, col=1)
    fig.add_scatter(x=XT, y=shape(SHAPE0, DRIVE0, XT), mode="lines",
                    line=dict(color=color, width=2.4), row=1, col=1)
    fig.add_scatter(x=PH, y=SIN_P, mode="lines",
                    line=dict(color=BLUE, width=1.2, dash="dash"), row=1, col=2)
    fig.add_scatter(x=PH, y=yp, mode="lines", line=dict(color=color, width=2.2),
                    row=1, col=2)
    xs, ys = stems(K, harmonics(y))
    fig.add_scatter(x=xs, y=ys, mode="lines", line=dict(color=color, width=2),
                    row=2, col=1)
    fig.update_xaxes(range=[-1, 1], title_text="x", fixedrange=True,
                     row=1, col=1)
    fig.update_yaxes(range=[-1.1, 1.1], title_text="f(x)",
                     fixedrange=True, row=1, col=1)
    fig.update_xaxes(range=[0, 2], title_text="Time (cycles)", fixedrange=True,
                     row=1, col=2)
    fig.update_yaxes(range=[-1.1, 1.1], title_text="f(x[n])", fixedrange=True,
                     row=1, col=2)
    fig.update_xaxes(range=[0.4, 25.6], dtick=2, tick0=1,
                     title_text="Harmonic k", fixedrange=True, row=2, col=1)
    fig.update_yaxes(range=[-80, 5], title_text="Level (dB)", fixedrange=True,
                     row=2, col=1)
    fig.update_layout(height=480)
    return fig


def controls(fig):
    name = widgets.Dropdown(description="Transfer function", value=SHAPE0,
                            options=list(SHAPES))
    drive = widgets.FloatLogSlider(description="Drive $g$", base=10,
                                   min=np.log10(DRIVE0) - 1.08, max=np.log10(30),
                                   step=0.01, value=DRIVE0, readout_format=".2f")
    source = widgets.Dropdown(description="Audio source", value=SOURCE0,
                              options=list(SOURCES))
    readout = widgets.HTML()

    # the defaults snapshot the arrays; the page's notebooks share one kernel
    def update(name, drive, shape=shape, harmonics=harmonics, stems=stems,
               SHAPES=SHAPES, SIN=SIN, SIN_P=SIN_P, XT=XT, K=K, readout=readout):
        h = harmonics(shape(name, drive, SIN))
        xs, ys = stems(K, h)
        color = SHAPES[name][2]
        with fig.batch_update():
            fig.data[1].y = shape(name, drive, XT)
            fig.data[3].y = shape(name, drive, SIN_P)
            fig.data[4].y = ys
            for i in (1, 3, 4):
                fig.data[i].line.color = color
        new = [int(k) for k, v in zip(K[1:], h[1:]) if v > -60]
        listed = ", ".join(map(str, new[:8])) + (", ..." if len(new) > 8 else "")
        text = (f"new harmonics above &minus;60 dB: {listed}" if new
                else "no new harmonics above &minus;60 dB: the output is still "
                     "(nearly) a pure sinusoid")
        formula = SHAPES[name][1].replace("<i>gx</i>", f"{drive:.2f}<i>x</i>")
        readout.value = (f"<span style='font-size:0.9em'><i>f</i>(<i>x</i>) = "
                         f"{formula} &nbsp;·&nbsp; {text}</span>")

    widgets.interactive_output(update, {"name": name, "drive": drive})

    # the audio card under the controls: the previous clip stays in place
    # while you drag (so the layout never jumps) and is swapped for the new
    # one when the pointer releases (keyboard nudges settle on a timer). It is
    # written through the Output's synced `outputs` trait, which works
    # outside a kernel message, where display() output has no destination
    out = widgets.Output()
    gate = icm_plotly.release_gate()   # pointer state: is a slider mid-drag?
    pending = []
    dirty = []

    def show_audio(x, SR=SR):
        audio = Audio(x.astype(np.float32), rate=SR, normalize=False)
        data, metadata = get_ipython().display_formatter.format(audio)
        # one assignment swaps the old card for the new one in place, so
        # the page never shows an empty card and nothing shifts
        out.outputs = ({"output_type": "display_data",
                        "data": data, "metadata": metadata},)

    def render(SOURCES=SOURCES, shape=shape):
        y = shape(name.value, drive.value, SOURCES[source.value])
        y = 0.5 * y / max(np.abs(y).max(), 1e-9)
        y[:441] *= np.linspace(0, 1, 441)
        y[-441:] *= np.linspace(1, 0, 441)
        show_audio(y)

    async def settle():
        await asyncio.sleep(0.25)
        pending.clear()
        if dirty and not gate.dragging:
            dirty.clear()
            render()

    def on_change(_):
        dirty.append(True)
        if pending:
            pending.pop().cancel()
        pending.append(asyncio.ensure_future(settle()))

    def on_release(change):
        if not change["new"] and dirty:
            if pending:
                pending.pop().cancel()
            dirty.clear()
            render()

    gate.observe(on_release, names="dragging")

    for s in (name, drive, source):
        s.observe(on_change, names="value")
    if not os.environ.get("ICM_BOOK_BUILD"):   # the build bakes no card
        render()
    return widgets.VBox([name, drive, source, readout, out, gate])

icm_plotly.show(figure, controls)